In [1]:
import pandas as pd
import ast

df = pd.read_csv("news_data.csv")

# drop yang kosong
df = df.dropna(subset=['article_text', 'tag'])

def parse_tags(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df['tags_list'] = df['tag'].apply(parse_tags)

print("Total data:", len(df))

Total data: 19812


In [2]:
from collections import Counter

counter = Counter()

for tags in df['tags_list']:
    counter.update(tags)

# filter tag muncul minimal 50x
top_tags = [tag for tag, c in counter.items() if c >= 50]

print("Jumlah tag terpilih:", len(top_tags))

Jumlah tag terpilih: 182


In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

model_embed = SentenceTransformer("indobenchmark/indobert-base-p1")

tag_embeddings = model_embed.encode(top_tags, show_progress_bar=True)

NUM_CLUSTERS = 10  # bisa kamu ubah (10–20 ideal)

kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42)
clusters = kmeans.fit_predict(tag_embeddings)

tag2cluster = dict(zip(top_tags, clusters))

W0429 21:51:54.860000 20952 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
No sentence-transformers model found with name indobenchmark/indobert-base-p1. Creating a new one with mean pooling.


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [4]:
cluster_dict = {}

for tag, c in tag2cluster.items():
    cluster_dict.setdefault(c, []).append(tag)

for k, v in cluster_dict.items():
    print(f"\nCluster {k}:")
    print(v[:20])  # tampilkan contoh


Cluster 7:
['hasto kristiyanto', 'tpn ganjar-mahfud', 'ganjar mahfud', 'ganjar pranowo', 'ganjar-mahfud', 'ganjar', 'polda metro jaya', 'ganjar capres', 'relawan ganjar', 'airlangga hartarto', 'detikjateng', 'budiman sudjatmiko', 'erick thohir', 'agus harimurti yudhoyono', 'ganjar pranowo capres', 'fx hadi rudyatmo', 'menko polhukam', 'susi pudjiastuti', 'tpn ganjar pranowo', 'cawapres ganjar']

Cluster 4:
['pemilu', 'politik', 'pilpres', 'pemilu 2024', 'kpu', 'prabowo', 'debat capres', 'pan', 'pilkada 2024', 'pdip', 'jokowi', 'pilpres 2024', 'capres 2024', 'golkar', 'bawaslu', 'capres', 'gerindra', 'ppp', 'partai demokrat', 'cawapres 2024']

Cluster 2:
['gibran rakabuming', 'round-up', 'prabowo-gibran', 'gibran rakabuming raka', 'gibran', 'tkn prabowo-gibran', 'kaesang pangarep', 'gibran cawapres prabowo', 'gibran cawapres', 'relawan gibran', 'prabowo gibran', 'andre rosiade', 'bambang pacul', 'puan maharani']

Cluster 8:
['sumut', 'jawa barat', 'berita jabar', 'politik sulsel', 'sul

In [5]:
label_map = {
    7: "ganjar_group",
    2: "prabowo_gibran_group",
    6: "anies_cak_imin_group",
    
    0: "tokoh_nasional",
    3: "tokoh_lain",
    
}

DROP_CLUSTERS = [1, 4, 5, 8, 9,0,3]  

In [6]:
def map_labels(tags):
    result = set()
    
    for t in tags:
        if t in tag2cluster:
            c = tag2cluster[t]
            
            if c in DROP_CLUSTERS:
                continue
            
            if c in label_map:
                result.add(label_map[c])
    
    return list(result)

df["labels"] = df["tags_list"].apply(map_labels)

# buang data tanpa label
df = df[df["labels"].map(len) > 0]

print("Data setelah filtering:", len(df))

Data setelah filtering: 10006


In [7]:
all_labels = sorted(list(set(l for labels in df["labels"] for l in labels)))
label2id = {l:i for i,l in enumerate(all_labels)}

def encode_labels(labels):
    vec = [0]*len(all_labels)
    for l in labels:
        vec[label2id[l]] = 1
    return vec

df["label_vec"] = df["labels"].apply(encode_labels)

print("Daftar label:", all_labels)

Daftar label: ['anies_cak_imin_group', 'ganjar_group', 'prabowo_gibran_group']


In [8]:
df_final = df[["article_text", "labels", "label_vec"]]

df_final.to_csv("news_multilabel.csv", index=False)

print("File berhasil dibuat: news_multilabel.csv")

File berhasil dibuat: news_multilabel.csv


In [9]:
print(df_final.head())

print("\nDistribusi label:")
from collections import Counter

label_counter = Counter()

for labels in df["labels"]:
    label_counter.update(labels)

print(label_counter)

                                        article_text                  labels  \
0  Sekjen PDIP sekaligus Sekretaris Tim Pemenang ...          [ganjar_group]   
1  Mantan Ketua Umum Pengurus Besar Nahdlatul Ula...  [prabowo_gibran_group]   
4  Ukraina marah usai Rusia menjadikan empat wila...  [prabowo_gibran_group]   
5  Sekretaris Tim Kampanye Nasional (TKN) Prabowo...  [prabowo_gibran_group]   
6  Cawapres nomor urut 2 Gibran Rakabuming Raka m...  [prabowo_gibran_group]   

   label_vec  
0  [0, 1, 0]  
1  [0, 0, 1]  
4  [0, 0, 1]  
5  [0, 0, 1]  
6  [0, 0, 1]  

Distribusi label:
Counter({'ganjar_group': 4802, 'prabowo_gibran_group': 3874, 'anies_cak_imin_group': 2803})
